In [ ]:
import numpy as np
import pandas as pd

In [ ]:
#Meta predictors (Ensemble modeller)
pure_meta_predictors = [
    #classic
    'REVEL_score_count',
    'REVEL_score_max',
    'REVEL_score_mean',
    'REVEL_score_std',
    'MetaLR_score',
    'MetaLR_score_was_na',
    'MetaSVM_score',
    'MetaSVM_score_was_na',
    'BayesDel_addAF_score',
    'BayesDel_addAF_score_was_na',
    'BayesDel_noAF_score',
    'BayesDel_noAF_score_was_na',
    'M-CAP_score',
    'M-CAP_score_was_na',
    'Eigen-phred_coding',
    'Eigen-phred_coding_was_na',
    'Eigen-raw_coding',
    'Eigen-raw_coding_was_na',
    'fathmm-XF_coding_score',
    'fathmm-XF_coding_score_was_na',
    'gMVP_score_count',
    'gMVP_score_max',
    'gMVP_score_mean',
    'gMVP_score_std',
    'CADD_phred']

new_dl_meta_like = [
    #new
    'AlphaMissense_rankscore',
    'AlphaMissense_rankscore_was_na',
    'AlphaMissense_score_count',
    'AlphaMissense_score_max',
    'AlphaMissense_score_mean',
    'AlphaMissense_score_std',
    'ESM1b_score_count',
    'ESM1b_score_max',
    'ESM1b_score_mean',
    'ESM1b_score_std',
    'MutPred2_score_count',
    'MutPred2_score_max',
    'MutPred2_score_mean',
    'MutPred2_score_std',
    'popEVE_score_count',
    'popEVE_score_max',
    'popEVE_score_mean',
    'popEVE_score_std',
    'DANN_score',
    'DANN_score_was_na'
]

#Fonksiyonel predictors (Tekil modeller)
functional_predictors = [
    'SIFT_score_count',
    'SIFT_score_max',
    'SIFT_score_mean',
    'SIFT_score_std',
    'MutationAssessor_score_count',
    'MutationAssessor_score_max',
    'MutationAssessor_score_mean',
    'MutationAssessor_score_std',
    'MutationTaster_score_count',
    'MutationTaster_score_max',
    'MutationTaster_score_mean',
    'MutationTaster_score_std',
    'Polyphen2_HDIV_score_count',
    'Polyphen2_HDIV_score_max',
    'Polyphen2_HDIV_score_mean',
    'Polyphen2_HDIV_score_std',
    'VEST4_score_count',
    'VEST4_score_max',
    'VEST4_score_mean',
    'VEST4_score_std',
    'MisFit_D_score_count',
    'MisFit_D_score_max',
    'MisFit_D_score_mean',
    'MisFit_D_score_std'
]

#Evrimsel korunmuşluk
conservation_scores = [
    #gerp tabanlı
    'GERP++_RS',
    'GERP++_RS_was_na',
    'GERP_92_mammals',
    'GERP_92_mammals_was_na',
    #olasılık tabanlı
    'phastCons100way_vertebrate',
    'phastCons100way_vertebrate_was_na',
    'phastCons470way_mammalian',
    'phastCons470way_mammalian_was_na',
    #evrimsel hız tabanlı
    'phyloP100way_vertebrate',
    'phyloP100way_vertebrate_was_na',
    'phyloP470way_mammalian',
    'phyloP470way_mammalian_was_na',
]

#Popülasyon Frekansı
population_score = [
    'gnomAD4.1_joint_AF',
    'gnomAD4.1_joint_AF_was_na',
    'gnomAD4.1_joint_POPMAX_AF',
    'gnomAD4.1_joint_POPMAX_AF_was_na',
    'dbNSFP_POPMAX_AF',
    'dbNSFP_POPMAX_AF_was_na'
]

# MANE
mane = [
    'MANE_Both',
    'MANE_Empty',
    'MANE_Plus_Clinical',
    'MANE_Select'
]

A_full = pure_meta_predictors+new_dl_meta_like+functional_predictors+conservation_scores+population_score+mane
B_no_meta_predictors = new_dl_meta_like+conservation_scores+population_score+mane
C_pure = conservation_scores+population_score+mane

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import random
import warnings

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression


# Uyarıları kapat (Temiz çıktı için)
warnings.filterwarnings('ignore')

# =========================
# 🔒 SEED & SETUP
# =========================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Veriyi yükle
df = pd.read_csv("all_train_test.csv")

TARGET = "label"
GROUP = "genename"

# Özellik setlerini tanımla (Buradaki listelerin dolu olduğunu varsayıyorum)
feature_sets = {
    "A": A_full,
    "B": B_no_meta_predictors,
    "C": C_pure
}

# =========================
# 🤖 MODELLER & YARDIMCILAR
# =========================
def get_models():
    return {
        "lightgbm": lgb.LGBMClassifier(random_state=SEED, verbosity=-1),
        "xgboost": xgb.XGBClassifier(random_state=SEED, eval_metric='logloss'),
        "logistic regression": LogisticRegression(random_state=SEED, n_jobs=-1)
    }



def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0, 1, 100)
    best_thr, best_f1 = 0.5, 0
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_thr = t
    return best_thr

# =========================
# 🔁 HESAPLAMA DÖNGÜSÜ
# =========================
gkf = GroupKFold(n_splits=5)
results = {}

for set_name, FEATURES in feature_sets.items():
    print(f"\n--- {set_name} SETİ ---")

    X = df[FEATURES]
    y = df[TARGET]
    groups = df[GROUP]
    results[set_name] = {}

    for model_name, model in get_models().items():
        thresholds = []

        for train_idx, val_idx in gkf.split(X, y, groups):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # Eksik veri tamamlama (Leakage-free)
            med = X_train.median()
            X_train = X_train.fillna(med)
            X_val = X_val.fillna(med)

            # Model eğitimi ve tahmin
            model.fit(X_train, y_train)
            y_prob = model.predict_proba(X_val)[:, 1]

            # En iyi eşiği bul
            thr = find_best_threshold(y_val, y_prob)
            thresholds.append(thr)

        # Ortalama eşiği kaydet ve ekrana yazdır
        final_thr = float(np.mean(thresholds))
        results[set_name][model_name] = final_thr
        print(f"{model_name}: {final_thr:.4f}")

# =========================
# 💾 KAYDET
# =========================
os.makedirs("thresholds", exist_ok=True)
with open("thresholds/thresholds_all.json", "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
#Meta predictors (Ensemble modeller)
pure_meta_predictors = [
    #classic
    'REVEL_score_count',
    'REVEL_score_max',
    'REVEL_score_mean',
    'REVEL_score_std',
    'MetaLR_score',
    'MetaLR_score_was_na',
    'MetaSVM_score',
    'MetaSVM_score_was_na',
    'BayesDel_addAF_score',
    'BayesDel_addAF_score_was_na',
    'BayesDel_noAF_score',
    'BayesDel_noAF_score_was_na',
    'M-CAP_score',
    'M-CAP_score_was_na',
    'Eigen-phred_coding',
    'Eigen-phred_coding_was_na',
    'Eigen-raw_coding',
    'Eigen-raw_coding_was_na',
    'fathmm-XF_coding_score',
    'fathmm-XF_coding_score_was_na',
    'gMVP_score_count',
    'gMVP_score_max',
    'gMVP_score_mean',
    'gMVP_score_std',
    'CADD_phred']

new_dl_meta_like = [
    #new
    'AlphaMissense_rankscore',
    'AlphaMissense_rankscore_was_na',
    'AlphaMissense_score_count',
    'AlphaMissense_score_max',
    'AlphaMissense_score_mean',
    'AlphaMissense_score_std',
    'ESM1b_score_count',
    'ESM1b_score_max',
    'ESM1b_score_mean',
    'ESM1b_score_std',
    'MutPred2_score_count',
    'MutPred2_score_max',
    'MutPred2_score_mean',
    'MutPred2_score_std',
    'popEVE_score_count',
    'popEVE_score_max',
    'popEVE_score_mean',
    'popEVE_score_std',
    'DANN_score',
    'DANN_score_was_na'
]

#Fonksiyonel predictors (Tekil modeller)
functional_predictors = [
    'SIFT_score_count',
    'SIFT_score_max',
    'SIFT_score_mean',
    'SIFT_score_std',
    'MutationAssessor_score_count',
    'MutationAssessor_score_max',
    'MutationAssessor_score_mean',
    'MutationAssessor_score_std',
    'MutationTaster_score_count',
    'MutationTaster_score_max',
    'MutationTaster_score_mean',
    'MutationTaster_score_std',
    'Polyphen2_HDIV_score_count',
    'Polyphen2_HDIV_score_max',
    'Polyphen2_HDIV_score_mean',
    'Polyphen2_HDIV_score_std',
    'VEST4_score_count',
    'VEST4_score_max',
    'VEST4_score_mean',
    'VEST4_score_std',
    'MisFit_D_score_count',
    'MisFit_D_score_max',
    'MisFit_D_score_mean',
    'MisFit_D_score_std'
]

#Evrimsel korunmuşluk
conservation_scores = [
    #gerp tabanlı
    'GERP++_RS',
    'GERP++_RS_was_na',
    'GERP_92_mammals',
    'GERP_92_mammals_was_na',
    #olasılık tabanlı
    'phastCons100way_vertebrate',
    'phastCons470way_mammalian',
    'phastCons470way_mammalian_was_na',
    #evrimsel hız tabanlı
    'phyloP100way_vertebrate',
    'phyloP470way_mammalian',
    'phyloP470way_mammalian_was_na',
]

#Popülasyon Frekansı
population_score = [
    'gnomAD4.1_joint_AF',
    'gnomAD4.1_joint_AF_was_na',
    'gnomAD4.1_joint_POPMAX_AF',
    'gnomAD4.1_joint_POPMAX_AF_was_na',
    'dbNSFP_POPMAX_AF',
    'dbNSFP_POPMAX_AF_was_na'
]

# MANE
mane = [
    'MANE_Both',
    'MANE_Empty',
    'MANE_Plus_Clinical',
    'MANE_Select'
]

A_full = pure_meta_predictors+new_dl_meta_like+functional_predictors+conservation_scores+population_score+mane
B_no_meta_predictors = new_dl_meta_like+conservation_scores+population_score+mane
C_pure = conservation_scores+population_score+mane

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import random
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')

# =========================
# 🔒 SEED
# =========================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# =========================
# 📂 VERİ
# =========================
df = pd.read_csv("cancer_train.csv")

TARGET = "label"

# Feature setlerin hazır olduğunu varsayıyorum
feature_sets = {
    "A": A_full,
    "B": B_no_meta_predictors,
    "C": C_pure
}

# =========================
# 🤖 MODEL TANIMLARI
# =========================
def get_models():
    return {
        "lightgbm": lgb.LGBMClassifier(random_state=SEED, verbosity=-1),
        "xgboost": xgb.XGBClassifier(random_state=SEED, eval_metric='logloss'),
        "logistic regression": LogisticRegression(random_state=SEED, n_jobs=-1)
    }


# =========================
# 🎯 THRESHOLD BULMA
# =========================
def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0, 1, 100)
    best_thr, best_f1 = 0.5, 0

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average='macro')

        if f1 > best_f1:
            best_f1 = f1
            best_thr = t

    return best_thr, best_f1

# =========================
# 🔁 ANA PIPELINE
# =========================
results = {}

for set_name, FEATURES in feature_sets.items():
    print(f"\n===== {set_name} SETİ =====")

    X = df[FEATURES]
    y = df[TARGET]

    # Stratified split (çok önemli)
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y
    )

    results[set_name] = {}

    for model_name, model in get_models().items():
        print(f"\nModel: {model_name}")

        # Model eğit
        model.fit(X_train, y_train)

        # Validation probability
        y_prob = model.predict_proba(X_val)[:, 1]

        # En iyi threshold
        best_thr, best_f1 = find_best_threshold(y_val, y_prob)

        results[set_name][model_name] = {
            "threshold": float(best_thr),
            "f1_score": float(best_f1)
        }

        print(f"Best Threshold: {best_thr:.4f}")
        print(f"Best F1 Score : {best_f1:.4f}")

# =========================
# 💾 SONUÇ KAYDET
# =========================
save_path = "thresholds/threshold_results_cancer.json"

with open(save_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSonuçlar kaydedildi: {save_path}")

In [ ]:
#Meta predictors (Ensemble modeller)
pure_meta_predictors = [
    #classic
    'REVEL_score_count',
    'REVEL_score_max',
    'REVEL_score_mean',
    'REVEL_score_std',
    'MetaLR_score',
    'MetaLR_score_was_na',
    'MetaSVM_score',
    'MetaSVM_score_was_na',
    'BayesDel_addAF_score',
    'BayesDel_addAF_score_was_na',
    'BayesDel_noAF_score',
    'BayesDel_noAF_score_was_na',
    'M-CAP_score',
    'M-CAP_score_was_na',
    'Eigen-phred_coding',
    'Eigen-phred_coding_was_na',
    'Eigen-raw_coding',
    'Eigen-raw_coding_was_na',
    'fathmm-XF_coding_score',
    'fathmm-XF_coding_score_was_na',
    'gMVP_score_count',
    'gMVP_score_max',
    'gMVP_score_mean',
    'gMVP_score_std',
    'CADD_phred']

new_dl_meta_like = [
    #new
    'AlphaMissense_rankscore',
    'AlphaMissense_rankscore_was_na',
    'AlphaMissense_score_count',
    'AlphaMissense_score_max',
    'AlphaMissense_score_mean',
    'AlphaMissense_score_std',
    'ESM1b_score_count',
    'ESM1b_score_max',
    'ESM1b_score_mean',
    'ESM1b_score_std',
    'MutPred2_score_count',
    'MutPred2_score_max',
    'MutPred2_score_mean',
    'MutPred2_score_std',
    'popEVE_score_count',
    'popEVE_score_max',
    'popEVE_score_mean',
    'popEVE_score_std',
    'DANN_score',
    'DANN_score_was_na'
]

#Fonksiyonel predictors (Tekil modeller)
functional_predictors = [
    'SIFT_score_count',
    'SIFT_score_max',
    'SIFT_score_mean',
    'SIFT_score_std',
    'MutationAssessor_score_count',
    'MutationAssessor_score_max',
    'MutationAssessor_score_mean',
    'MutationAssessor_score_std',
    'MutationTaster_score_count',
    'MutationTaster_score_max',
    'MutationTaster_score_mean',
    'MutationTaster_score_std',
    'Polyphen2_HDIV_score_count',
    'Polyphen2_HDIV_score_max',
    'Polyphen2_HDIV_score_mean',
    'Polyphen2_HDIV_score_std',
    'VEST4_score_count',
    'VEST4_score_max',
    'VEST4_score_mean',
    'VEST4_score_std',
    'MisFit_D_score_count',
    'MisFit_D_score_max',
    'MisFit_D_score_mean',
    'MisFit_D_score_std'
]

#Evrimsel korunmuşluk
conservation_scores = [
    #gerp tabanlı
    'GERP++_RS',
    'GERP++_RS_was_na',
    'GERP_92_mammals',
    'GERP_92_mammals_was_na',
    #olasılık tabanlı
    'phastCons100way_vertebrate',
    'phastCons470way_mammalian',
    #evrimsel hız tabanlı
    'phyloP100way_vertebrate',
    'phyloP470way_mammalian',
    'phyloP470way_mammalian_was_na',
]

#Popülasyon Frekansı
population_score = [
    'gnomAD4.1_joint_AF',
    'gnomAD4.1_joint_AF_was_na',
    'gnomAD4.1_joint_POPMAX_AF',
    'gnomAD4.1_joint_POPMAX_AF_was_na',
    'dbNSFP_POPMAX_AF',
    'dbNSFP_POPMAX_AF_was_na'
]

# MANE
mane = [
    'MANE_Empty',
    'MANE_Select'
]

A_full = pure_meta_predictors+new_dl_meta_like+functional_predictors+conservation_scores+population_score+mane
B_no_meta_predictors = new_dl_meta_like+conservation_scores+population_score+mane
C_pure = conservation_scores+population_score+mane

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import random
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

# =========================
# 🔒 SEED
# =========================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# =========================
# 📂 VERİ
# =========================
df = pd.read_csv("pah_train.csv")

TARGET = "label"

# Feature setlerin hazır olduğunu varsayıyorum
feature_sets = {
    "A": A_full,
    "B": B_no_meta_predictors,
    "C": C_pure
}

# =========================
# 🤖 MODEL TANIMLARI
# =========================
def get_models():
    return {
        "lightgbm": lgb.LGBMClassifier(random_state=SEED, verbosity=-1),
        "xgboost": xgb.XGBClassifier(random_state=SEED, eval_metric='logloss'),
        "logistic regression": LogisticRegression(random_state=SEED, n_jobs=-1)
    }


# =========================
# 🎯 THRESHOLD BULMA
# =========================
def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0, 1, 100)
    best_thr, best_f1 = 0.5, 0

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average='macro')

        if f1 > best_f1:
            best_f1 = f1
            best_thr = t

    return best_thr, best_f1

# =========================
# 🔁 ANA PIPELINE
# =========================
results = {}

for set_name, FEATURES in feature_sets.items():
    print(f"\n===== {set_name} SETİ =====")

    X = df[FEATURES]
    y = df[TARGET]

    # Stratified split (çok önemli)
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y
    )

    results[set_name] = {}

    for model_name, model in get_models().items():
        print(f"\nModel: {model_name}")

        # Model eğit
        model.fit(X_train, y_train)

        # Validation probability
        y_prob = model.predict_proba(X_val)[:, 1]

        # En iyi threshold
        best_thr, best_f1 = find_best_threshold(y_val, y_prob)

        results[set_name][model_name] = {
            "threshold": float(best_thr),
            "f1_score": float(best_f1)
        }

        print(f"Best Threshold: {best_thr:.4f}")
        print(f"Best F1 Score : {best_f1:.4f}")

# =========================
# 💾 SONUÇ KAYDET
# =========================
save_path = "thresholds/threshold_results_pah.json"

with open(save_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSonuçlar kaydedildi: {save_path}")

In [ ]:
#Meta predictors (Ensemble modeller)
pure_meta_predictors = [
    #classic
    'REVEL_score_count',
    'REVEL_score_max',
    'REVEL_score_mean',
    'REVEL_score_std',
    'MetaLR_score',
    'MetaLR_score_was_na',
    'MetaSVM_score',
    'MetaSVM_score_was_na',
    'BayesDel_addAF_score',
    'BayesDel_addAF_score_was_na',
    'BayesDel_noAF_score',
    'BayesDel_noAF_score_was_na',
    'M-CAP_score',
    'M-CAP_score_was_na',
    'Eigen-phred_coding',
    'Eigen-phred_coding_was_na',
    'Eigen-raw_coding',
    'Eigen-raw_coding_was_na',
    'fathmm-XF_coding_score',
    'fathmm-XF_coding_score_was_na',
    'gMVP_score_count',
    'gMVP_score_max',
    'gMVP_score_mean',
    'gMVP_score_std',
    'CADD_phred']

new_dl_meta_like = [
    #new
    'AlphaMissense_rankscore',
    'AlphaMissense_rankscore_was_na',
    'AlphaMissense_score_count',
    'AlphaMissense_score_max',
    'AlphaMissense_score_mean',
    'AlphaMissense_score_std',
    'ESM1b_score_count',
    'ESM1b_score_max',
    'ESM1b_score_mean',
    'ESM1b_score_std',
    'MutPred2_score_count',
    'MutPred2_score_max',
    'MutPred2_score_mean',
    'MutPred2_score_std',
    'popEVE_score_count',
    'popEVE_score_max',
    'popEVE_score_mean',
    'popEVE_score_std',
    'DANN_score'
    ]

#Fonksiyonel predictors (Tekil modeller)
functional_predictors = [
    'SIFT_score_count',
    'SIFT_score_max',
    'SIFT_score_mean',
    'SIFT_score_std',
    'MutationAssessor_score_count',
    'MutationAssessor_score_max',
    'MutationAssessor_score_mean',
    'MutationAssessor_score_std',
    'MutationTaster_score_count',
    'MutationTaster_score_max',
    'MutationTaster_score_mean',
    'MutationTaster_score_std',
    'Polyphen2_HDIV_score_count',
    'Polyphen2_HDIV_score_max',
    'Polyphen2_HDIV_score_mean',
    'Polyphen2_HDIV_score_std',
    'VEST4_score_count',
    'VEST4_score_max',
    'VEST4_score_mean',
    'VEST4_score_std',
    'MisFit_D_score_count',
    'MisFit_D_score_max',
    'MisFit_D_score_mean',
    'MisFit_D_score_std'
]

#Evrimsel korunmuşluk
conservation_scores = [
    #gerp tabanlı
    'GERP++_RS',
    'GERP_92_mammals',
    'GERP_92_mammals_was_na',
    #olasılık tabanlı
    'phastCons100way_vertebrate',
    'phastCons470way_mammalian',
    #evrimsel hız tabanlı
    'phyloP100way_vertebrate',
    'phyloP470way_mammalian'
]

#Popülasyon Frekansı
population_score = [
    'gnomAD4.1_joint_AF',
    'gnomAD4.1_joint_AF_was_na',
    'gnomAD4.1_joint_POPMAX_AF',
    'gnomAD4.1_joint_POPMAX_AF_was_na',
    'dbNSFP_POPMAX_AF',
    'dbNSFP_POPMAX_AF_was_na'
]

# MANE
mane = [
    'MANE_Empty',
    'MANE_Select'
]

A_full = pure_meta_predictors+new_dl_meta_like+functional_predictors+conservation_scores+population_score+mane
B_no_meta_predictors = new_dl_meta_like+conservation_scores+population_score+mane
C_pure = conservation_scores+population_score+mane

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import random
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

# =========================
# 🔒 SEED
# =========================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# =========================
# 📂 VERİ
# =========================
df = pd.read_csv("cftr_train.csv")

TARGET = "label"

# Feature setlerin hazır olduğunu varsayıyorum
feature_sets = {
    "A": A_full,
    "B": B_no_meta_predictors,
    "C": C_pure
}

# =========================
# 🤖 MODEL TANIMLARI
# =========================
def get_models():
    return {
        "lightgbm": lgb.LGBMClassifier(random_state=SEED, verbosity=-1),
        "xgboost": xgb.XGBClassifier(random_state=SEED, eval_metric='logloss'),
        "logistic regression": LogisticRegression(random_state=SEED, n_jobs=-1)
    }


# =========================
# 🎯 THRESHOLD BULMA
# =========================
def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0, 1, 100)
    best_thr, best_f1 = 0.5, 0

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, average='macro')

        if f1 > best_f1:
            best_f1 = f1
            best_thr = t

    return best_thr, best_f1

# =========================
# 🔁 ANA PIPELINE
# =========================
results = {}

for set_name, FEATURES in feature_sets.items():
    print(f"\n===== {set_name} SETİ =====")

    X = df[FEATURES]
    y = df[TARGET]

    # Stratified split (çok önemli)
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y
    )

    results[set_name] = {}

    for model_name, model in get_models().items():
        print(f"\nModel: {model_name}")

        # Model eğit
        model.fit(X_train, y_train)

        # Validation probability
        y_prob = model.predict_proba(X_val)[:, 1]

        # En iyi threshold
        best_thr, best_f1 = find_best_threshold(y_val, y_prob)

        results[set_name][model_name] = {
            "threshold": float(best_thr),
            "f1_score": float(best_f1)
        }

        print(f"Best Threshold: {best_thr:.4f}")
        print(f"Best F1 Score : {best_f1:.4f}")

# =========================
# 💾 SONUÇ KAYDET
# =========================
save_path = "thresholds/threshold_results_cftr.json"

with open(save_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSonuçlar kaydedildi: {save_path}")